In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')
df.show()

# **_`Trimming`_**

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

**_Normalize maintenance Flag to Boolean_**

In [0]:
df.withColumn(
    'maintenance',
    F.when(F.upper(col("Maintenance")) == 'YES', F.lit(True))
     .when(F.upper(col('maintenance')) =='NO', F.lit(False))
     .otherwise(None)
)

# **_Renaming Columns_**

In [0]:
RENAME_MAP ={
    'id': 'category_id',
    'cat':'category',
    'subcat':'subcategory',
    'maintenance':'maintenance_flag'
}

for old_name , new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

# _**Writing to Silver Table**_

In [0]:
df.write.mode("overwrite").format('delta').saveAsTable("silver.erp_px_cat_g1v2")

In [0]:
%sql
select * from silver.erp_px_cat_g1v2 limit 10